# Modelo Robusto de Subenquadramento — PNCP

**Objetivo:** construir um **classificador reprodutível** que, dado o objeto de
um contrato público, diga se ele é de **engenharia/obras** ou **não** — para
identificar contratos rotulados como "serviços gerais" que deveriam ser
engenharia (subenquadramento, Lei 14.133/2021).

## Estratégia em 10 etapas

| Etapa | O que faz | Insumo |
|---|---|---|
| 0 | Setup (libs, Drive, LLM, paths) | — |
| 1 | EDA + baseline kNN supervisionado (só para ter ideia) | rótulos existentes |
| 2 | **Clusterização não-supervisionada** (poucos clusters) | SBERT, sem rótulos |
| 3 | **Revisão MANUAL dos clusters** (conhecimento do engenheiro) | CSV editável |
| 4 | LLM descreve perfil "engenharia" vs "não engenharia" | clusters certeiros |
| 5 | Treina modelos supervisionados (RF, LR, SVC) | contratos certeiros |
| 6 | Classifica clusters duvidosos com o melhor modelo | melhor modelo |
| 7 | **Visualização UMAP 2D** — duvidoso perto de certeiro indica classe | embeddings |
| 8 | Veredito final do LLM nos classificados como engenharia | LLM |
| 9 | **Análise de rito** nos PDFs já no Drive (TR/Edital/Aditivos) | PDFs cacheados |
| 10 | Exportação consolidada (.md + CSVs + gráficos) | tudo acima |

**Ideia central:** a clusterização desacoplada dos rótulos + revisão humana
gera **rótulos limpos** (clusters certeiros) que treinam um classificador
robusto. Os duvidosos são classificados pelo melhor modelo e visualizados
no espaço semântico para validação visual.

**Pré-requisitos:**
- `contratos.parquet` já no Drive
- (opcional) PDFs cacheados de rodadas anteriores no Drive
- GPU recomendada (SBERT + UMAP)

## 0. Setup

Bibliotecas, montagem do Drive, escolha do LLM (Gemini ou Ollama local) e
definição dos paths.

In [ ]:
!pip install -q sentence-transformers nltk scikit-learn pyarrow pandas
!pip install -q umap-learn matplotlib seaborn
!pip install -q google-generativeai
!pip install -q pymupdf

In [ ]:
import os, re, time, json, unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for pkg in ('punkt', 'punkt_tab', 'stopwords', 'rslp'):
    nltk.download(pkg, quiet=True)
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    silhouette_score, ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 180)

In [ ]:
# Drive + paths
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

CAMINHO_PARQUET = '/content/drive/MyDrive/PNCP_TCC/dados/coleta/contratos.parquet'
PASTA_RESULTADOS = '/content/drive/MyDrive/PNCP_TCC/resultados_modelo'
os.makedirs(PASTA_RESULTADOS, exist_ok=True)

print(f'Resultados em: {PASTA_RESULTADOS}')

In [ ]:
# LLM (Gemini com auto-detecção). Para Ollama local, troque o llm_task abaixo.
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

_PREFERIDOS = ['gemini-2.5-flash-lite', 'gemini-2.0-flash-lite',
                'gemini-2.0-flash', 'gemini-2.5-flash', 'gemini-flash-latest']

def detectar_modelo():
    try:
        disp = [m.name.split('/')[-1] for m in genai.list_models()
                if 'generateContent' in getattr(m, 'supported_generation_methods', [])]
    except Exception:
        return 'gemini-2.0-flash'
    for p in _PREFERIDOS:
        if p in disp:
            return p
    return next((m for m in disp if 'flash' in m), 'gemini-2.0-flash')

MODELO_LLM = detectar_modelo()
INTERVALO_LLM_SEG = 4   # 5s entre chamadas (sleep para ficar abaixo do RPM free)

def llm_task(model, system, prompt, max_tokens=800, temperatura=0.2,
              max_tentativas=3):
    for tent in range(max_tentativas):
        try:
            gm = genai.GenerativeModel(model_name=model,
                generation_config={'temperature': temperatura,
                                   'max_output_tokens': max_tokens})
            resp = gm.generate_content([system, prompt])
            return (resp.text or '').strip()
        except Exception as e:
            msg = str(e)
            if ('429' in msg or 'quota' in msg.lower()) and tent < max_tentativas - 1:
                time.sleep(20 * (2 ** tent))
                continue
            print(f'[llm] erro: {msg[:140]}')
            return None
    return None

def extrair_json(txt):
    if not txt:
        return None
    t = txt.replace('```json', '').replace('```', '')
    m = re.search(r'\{.*\}', t, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

print(f'LLM: {MODELO_LLM}')

In [ ]:
# ── Pré-processamento de texto ──────────────────────────────────────────
STOP_PT = set(nltk.corpus.stopwords.words('portuguese'))
STOP_DOMINIO = {
    'contratacao','contratacoes','contratar','contrato','contratos',
    'servico','servicos','prestacao','prestar','prestados','prestada',
    'empresa','empresas','especializada','especializado',
    'fornecimento','fornecer','aquisicao','aquisicoes','fornecedor',
    'registro','preco','precos','atender','atendimento',
    'executar','execucao','realizacao','realizar','objeto','objetos',
    'conforme','demanda','demandas','referente','pregao','licitacao',
    'edital','editais','ata','atas','municipio','municipal',
    'estadual','federal','lote','lotes','item','itens',
}
STOP_TUDO = STOP_PT | STOP_DOMINIO
STEMMER = RSLPStemmer()

def _normalizar(s):
    s = str(s).lower()
    return ''.join(c for c in unicodedata.normalize('NFKD', s)
                    if not unicodedata.combining(c))

def meu_tokenizador(doc):
    s = _normalizar(doc)
    toks = word_tokenize(s, language='portuguese')
    return [STEMMER.stem(w) for w in toks
            if w not in STOP_TUDO and w.isalnum() and not w.isdigit() and len(w) > 2]

# Limpa prefixos burocráticos para o SBERT — sem isso, "CONTRATAÇÃO DE EMPRESA
# ESPECIALIZADA PARA PRESTAÇÃO DE SERVIÇOS DE..." idêntico em quase todo
# contrato arrasta a similaridade para cima e arruina a discriminação.
_RX_BP = [re.compile(p, re.IGNORECASE) for p in [
    r'^\s*registro\s+de\s+pre[çc]os?\s+(para\s+a?\s*)?',
    r'^\s*contrata[çc][ãa]o\s+(de\s+)?(empresa\s+)?(especializada\s+)?'
    r'(em\s+|para\s+(o?\s+|a?\s+)?(presta[çc][ãa]o\s+(de\s+)?)?(servi[çc]os?\s+(de\s+)?)?)?',
    r'^\s*contrata[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?',
    r'^\s*presta[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?',
    r'^\s*aquisi[çc][ãa]o\s+de\s+',
    r'^\s*fornecimento\s+de\s+',
    r'^\s*execu[çc][ãa]o\s+de\s+(servi[çc]os?\s+(de\s+)?)?',
    r'^\s*loca[çc][ãa]o\s+de\s+',
]]

def limpar_boilerplate(t):
    if not isinstance(t, str):
        return ''
    s = t.strip()
    for _ in range(3):
        for rx in _RX_BP:
            s = rx.sub('', s, count=1)
    return re.sub(r'\s+', ' ', s).strip()

In [ ]:
# ── Carrega base ───────────────────────────────────────────────────────
df = pd.read_parquet(CAMINHO_PARQUET)
# Detecta UF
COL_UF = next((c for c in df.columns
               if c.lower() in ('uf','siglauf','ufsigla','unidadefederativa')), None)
if COL_UF and COL_UF != 'uf':
    df = df.rename(columns={COL_UF: 'uf'})

cols = [c for c in ['numeroControlePNCP','objeto','rotulo','razaoSocialOrgao',
                     'municipioNome','valor','uf','numeroControlePNCPCompra',
                     'sequencialCompra','anoCompra'] if c in df.columns]
df = df[cols].copy()
df = df.rename(columns={'objeto': 'text'}).dropna(subset=['text'])
df = df[df['text'].str.len() > 20].reset_index(drop=True)
print(f'Total: {len(df):,} contratos')
print(df['rotulo'].value_counts().to_string())

# Amostragem para acelerar (ajuste à vontade — em GPU paga, use tudo)
N_AMOSTRA = 30_000
if len(df) > N_AMOSTRA:
    df = df.sample(n=N_AMOSTRA, random_state=42).reset_index(drop=True)
print(f'\nAmostra de trabalho: {len(df):,}')

## 1. EDA + baseline supervisionado (para ter ideia da base)

Aqui usamos os **rótulos existentes** apenas para entender a base e ter um
ponto de comparação. Esta etapa **não influencia** a clusterização da Etapa 2.

In [ ]:
# 1.1 Distribuição dos rótulos
fig, ax = plt.subplots(figsize=(7, 3.5))
cont = df['rotulo'].value_counts()
cores = {'engenharia': '#d62728', 'obras': '#ff7f0e', 'geral': '#9aa0a6'}
ax.bar(cont.index, cont.values, color=[cores.get(c, '#1f77b4') for c in cont.index])
for c, v in zip(cont.index, cont.values):
    ax.text(c, v, f'{v:,}', ha='center', va='bottom')
ax.set_title(f'Distribuição dos rótulos atuais (n={len(df):,})')
ax.set_ylabel('nº contratos')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/01_distribuicao_rotulos.png', dpi=120,
            bbox_inches='tight')
plt.show()

In [ ]:
# 1.2 Top termos TF-IDF (unigramas) e top bigramas
print('Construindo TF-IDF unigramas + bigramas...')
vsm1 = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                       min_df=5, ngram_range=(1, 1), max_features=20000)
X1 = vsm1.fit_transform(df['text'])
top1 = (pd.DataFrame({'termo': vsm1.get_feature_names_out(),
                      'tfidf_sum': X1.toarray().sum(axis=0)})
        .sort_values('tfidf_sum', ascending=False).head(30))
print('\nTop 30 termos:')
print(top1.to_string(index=False))

In [ ]:
vsm2 = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                       min_df=5, ngram_range=(2, 2), max_features=15000)
X2 = vsm2.fit_transform(df['text'])
top2 = (pd.DataFrame({'bigrama': vsm2.get_feature_names_out(),
                      'tfidf_sum': X2.toarray().sum(axis=0)})
        .sort_values('tfidf_sum', ascending=False).head(30))
print('Top 30 bigramas (vocabulário composto):')
print(top2.to_string(index=False))

In [ ]:
# 1.3 Baseline kNN supervisionado — só para ter um número de referência
# Não é o modelo final; é só para sabermos onde estamos partindo.
print('Treinando kNN baseline (TF-IDF + k=5)...')
y_full = df['rotulo'].values
X_tr, X_te, y_tr, y_te = train_test_split(X1, y_full, test_size=0.2,
                                            random_state=42, stratify=y_full)
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine', n_jobs=-1)
knn.fit(X_tr, y_tr)
pred = knn.predict(X_te)
print(f'\nBaseline kNN — F1 macro: {f1_score(y_te, pred, average="macro"):.3f}')
print(classification_report(y_te, pred, zero_division=0))
print('\n→ Este F1 é só REFERÊNCIA. O modelo final virá nas etapas 5–6.')

## 2. Clusterização não-supervisionada (sem usar rótulos)

**Princípio:** ignoramos os rótulos `engenharia`/`obras`/`geral` aqui. A
clusterização agrupa contratos por **similaridade semântica** (SBERT). O
resultado é independente dos rótulos atuais — pode até revelá-los como
errados, que é o ponto.

**Por que SBERT em vez de TF-IDF puro?** SBERT capta sinônimos e contexto.
"reparo no telhado" e "manutenção de cobertura" vão para o mesmo cluster
mesmo sem palavras em comum.

**Por que poucos clusters (k=15)?** Para a revisão humana da Etapa 3 ser
viável. Cada cluster representa uma "família" semântica que você marca
como inteira.

In [ ]:
# 2.1 Embeddings SBERT (com limpeza de boilerplate)
from sentence_transformers import SentenceTransformer

print('Carregando SBERT...')
sbert = SentenceTransformer('distiluse-base-multilingual-cased-v1')

textos_limpos = df['text'].map(limpar_boilerplate).tolist()
print('Gerando embeddings (pode levar alguns minutos)...')
X_emb = sbert.encode(textos_limpos, show_progress_bar=True, batch_size=64,
                     convert_to_numpy=True, normalize_embeddings=True)
print(f'Embeddings: {X_emb.shape}')

In [ ]:
# 2.2 KMeans com poucos clusters
K = 15

print(f'Rodando KMeans com k={K}...')
km = KMeans(n_clusters=K, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X_emb)

# Silhouette em amostra (cálculo total é caro)
amostra = np.random.RandomState(42).choice(len(df), size=min(5000, len(df)),
                                            replace=False)
sil = silhouette_score(X_emb[amostra], df['cluster'].values[amostra],
                       metric='cosine')
print(f'Silhouette (amostra 5k): {sil:.3f}')
print('\nDistribuição dos clusters:')
print(df['cluster'].value_counts().sort_index().to_string())

In [ ]:
# 2.3 Descritores por cluster: top termos TF-IDF + dist. dos rótulos atuais
# A distribuição dos rótulos é mostrada APENAS como auxílio à revisão humana.
# Não influencia a clusterização (já feita).

vsm_desc = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                           min_df=3, ngram_range=(1, 2), max_features=30000)
X_desc = vsm_desc.fit_transform(df['text'])

descritores = []
for c in sorted(df['cluster'].unique()):
    mask = df['cluster'] == c
    sub = df[mask]
    # Top termos do cluster
    soma = X_desc[mask.values].sum(axis=0).A1
    top_idx = soma.argsort()[::-1][:8]
    top_termos = [vsm_desc.get_feature_names_out()[i] for i in top_idx]
    # Distribuição dos rótulos atuais
    dist = sub['rotulo'].value_counts().to_dict()
    descritores.append({
        'cluster': c,
        'n_docs': len(sub),
        'top_termos': ', '.join(top_termos),
        'n_engenharia': dist.get('engenharia', 0),
        'n_obras': dist.get('obras', 0),
        'n_geral': dist.get('geral', 0),
        'pct_eng_obras': (dist.get('engenharia', 0) + dist.get('obras', 0))
                          / len(sub),
    })
df_desc = pd.DataFrame(descritores).sort_values('pct_eng_obras', ascending=False)
print('Descritores dos clusters (ordenados por % eng+obras):\n')
print(df_desc.to_string(index=False))

## 3. Revisão MANUAL dos clusters (conhecimento do engenheiro)

**Esta é a etapa-chave.** Você, como engenheiro, vai marcar cada cluster
como uma destas 3 categorias:

| Marcação | Quando usar |
|---|---|
| `eng_obra` | **Certeza** de que é engenharia ou obra |
| `nao` | **Certeza** de que NÃO é engenharia/obra (limpeza, vigilância, alimentação, fornecimento de bens, evento, etc.) |
| `duvidoso` | Cluster ambíguo ou misturado — vai para o classificador da Etapa 6 |

A distribuição dos rótulos atuais (`n_engenharia`/`n_obras`/`n_geral`) é
mostrada como **auxílio** — você pode discordar dela. Os top termos e as
amostras de objetos abaixo são a sua base real de decisão.

In [ ]:
# 3.1 Gera arquivo de revisão (CSV no Drive) com tudo que você precisa
import csv as _csv

N_AMOSTRAS = 8   # nº de objetos de exemplo por cluster

linhas = []
for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster'] == c]
    desc_row = df_desc[df_desc['cluster'] == c].iloc[0]
    amostras = sub['text'].sample(min(N_AMOSTRAS, len(sub)),
                                   random_state=c).tolist()
    linha = {
        'cluster': c,
        'n_docs': desc_row['n_docs'],
        'top_termos': desc_row['top_termos'],
        'n_engenharia': desc_row['n_engenharia'],
        'n_obras': desc_row['n_obras'],
        'n_geral': desc_row['n_geral'],
        'pct_eng_obras': round(desc_row['pct_eng_obras'], 3),
        'rotulo_humano': '',   # <- VOCÊ PREENCHE: eng_obra | nao | duvidoso
    }
    for i, a in enumerate(amostras, 1):
        linha[f'amostra_{i}'] = str(a)[:300]
    linhas.append(linha)

caminho_revisao = f'{PASTA_RESULTADOS}/03_revisao_clusters.csv'
pd.DataFrame(linhas).to_csv(caminho_revisao, index=False, encoding='utf-8-sig')
print(f'CSV de revisão criado em:\n  {caminho_revisao}\n')
print('→ ABRA o CSV no Excel/Sheets, preencha a coluna "rotulo_humano" para')
print('  CADA cluster com: eng_obra | nao | duvidoso')
print('  Salve e rode a próxima célula.')

### ⏸ PAUSE — abra o CSV, preencha `rotulo_humano`, salve, e continue

A próxima célula carrega o CSV de volta e valida os rótulos.

In [ ]:
# 3.2 Carrega o CSV preenchido
df_revisao = pd.read_csv(caminho_revisao)
df_revisao['rotulo_humano'] = (df_revisao['rotulo_humano']
                                .fillna('').str.strip().str.lower())

# Valida
validos = {'eng_obra', 'nao', 'duvidoso'}
invalidos = df_revisao[~df_revisao['rotulo_humano'].isin(validos)]
if len(invalidos):
    print(f'⚠ {len(invalidos)} cluster(s) sem rótulo válido (esperado: '
          f'{validos}):')
    print(invalidos[['cluster', 'rotulo_humano']].to_string(index=False))
    print('\nPreencha o CSV e rode esta célula de novo.')
else:
    cont = df_revisao['rotulo_humano'].value_counts()
    print('Distribuição da rotulação humana:')
    print(cont.to_string())
    # Propaga para o DataFrame principal
    mapa = dict(zip(df_revisao['cluster'], df_revisao['rotulo_humano']))
    df['rotulo_humano'] = df['cluster'].map(mapa)
    print('\nContratos por veredito humano (via cluster):')
    print(df['rotulo_humano'].value_counts().to_string())

## 4. LLM descreve perfil de "engenharia" e "não engenharia"

A partir dos clusters que **você** marcou como certeiros, o LLM lê amostras
de objetos e gera duas descrições estruturadas dos perfis. Isso serve para
documentar e auditar a base de rotulação humana.

In [ ]:
SYS_PERFIL = '''Você é um especialista em contratações públicas brasileiras
(Lei 14.133/2021). Vai ler uma lista de OBJETOS de contratos que foram
classificados (por revisão humana) como pertencentes a um determinado grupo.

Sua tarefa: descrever o PERFIL desse grupo em JSON, identificando padrões
linguísticos e semânticos.

Considere apenas engenharia (CREA/ART) e obras. Arquitetura (CAU/RRT) NÃO
entra.

Responda APENAS no JSON:
{"resumo": "1-2 frases sintéticas",
 "padroes_lexicais": ["palavra/expressão recorrente 1", "...", "..."],
 "tipos_servico": ["tipo de serviço típico 1", "...", "..."],
 "indicadores_fortes": ["sinal que com certeza indica essa classe"]}'''

def perfil_llm(rotulo_humano, n_amostras=15):
    sub = df[df['rotulo_humano'] == rotulo_humano].sample(
        min(n_amostras, (df['rotulo_humano'] == rotulo_humano).sum()),
        random_state=42)
    objetos = '\n'.join(f'- {str(o)[:250]}' for o in sub['text'])
    prompt = f'Grupo: "{rotulo_humano}"\n\nObjetos:\n{objetos}\n\nDescreva o perfil.'
    resp = llm_task(MODELO_LLM, SYS_PERFIL, prompt)
    return extrair_json(resp) or {}

perfil_eng = perfil_llm('eng_obra')
time.sleep(INTERVALO_LLM_SEG)
perfil_nao = perfil_llm('nao')

print('═' * 60)
print('PERFIL — eng_obra')
print('═' * 60)
print(json.dumps(perfil_eng, ensure_ascii=False, indent=2))
print()
print('═' * 60)
print('PERFIL — nao engenharia')
print('═' * 60)
print(json.dumps(perfil_nao, ensure_ascii=False, indent=2))

with open(f'{PASTA_RESULTADOS}/04_perfis_llm.json', 'w', encoding='utf-8') as f:
    json.dump({'eng_obra': perfil_eng, 'nao': perfil_nao}, f,
              ensure_ascii=False, indent=2)

## 5. Treina modelos supervisionados nos clusters certeiros

Conjunto de treino = contratos dos clusters marcados como `eng_obra` ou `nao`
pela revisão humana. Os duvidosos ficam de fora — eles vão ser classificados
na Etapa 6.

**Modelos comparados:** Random Forest, Logistic Regression, LinearSVC
(calibrado). Métrica: F1 macro com holdout 80/20.

In [ ]:
# 5.1 Monta treino: só certeiros
mask_certeiro = df['rotulo_humano'].isin(['eng_obra', 'nao'])
df_certeiro = df[mask_certeiro].reset_index(drop=True)
X_cert = X_emb[mask_certeiro.values]
y_cert = (df_certeiro['rotulo_humano'] == 'eng_obra').astype(int).values

print(f'Certeiros: {len(df_certeiro):,} contratos | '
      f'eng_obra={y_cert.sum():,} | nao={(y_cert==0).sum():,}')

X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X_cert, y_cert, np.arange(len(y_cert)), test_size=0.2,
    random_state=42, stratify=y_cert)
print(f'Treino: {len(y_tr):,} | Teste: {len(y_te):,}')

In [ ]:
# 5.2 Treina e compara 3 modelos
modelos = {
    'rf': RandomForestClassifier(n_estimators=200, max_depth=20,
                                  class_weight='balanced',
                                  n_jobs=-1, random_state=42),
    'lr': LogisticRegression(max_iter=3000, class_weight='balanced',
                              C=1.0, n_jobs=-1, random_state=42),
    'svc': CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', random_state=42, max_iter=5000),
        cv=3),
}

resultados = {}
for nome, m in modelos.items():
    print(f'\nTreinando {nome}...')
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)
    f1 = f1_score(y_te, pred, average='macro')
    f1_eng = f1_score(y_te, pred, labels=[1], average='macro', zero_division=0)
    resultados[nome] = {'modelo': m, 'f1_macro': f1, 'f1_eng': f1_eng,
                        'pred': pred}
    print(f'  F1 macro: {f1:.3f} | F1 eng_obra: {f1_eng:.3f}')

print('\n══ Ranking por F1 macro ══')
for nome, r in sorted(resultados.items(), key=lambda x: -x[1]['f1_macro']):
    print(f'  {nome}: {r["f1_macro"]:.3f}')

melhor_nome = max(resultados, key=lambda n: resultados[n]['f1_macro'])
melhor = resultados[melhor_nome]['modelo']
print(f'\n🏆 Melhor modelo: {melhor_nome} '
      f'(F1 macro = {resultados[melhor_nome]["f1_macro"]:.3f})')

In [ ]:
# 5.3 Matriz de confusão do melhor modelo
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (nome, r) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_te, r['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['nao', 'eng_obra'],
                yticklabels=['nao', 'eng_obra'])
    ax.set_title(f'{nome} — F1={r["f1_macro"]:.3f}')
    ax.set_xlabel('predito'); ax.set_ylabel('real')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/05_matrizes_confusao.png', dpi=120,
            bbox_inches='tight')
plt.show()

# Salva métricas
salvar = {n: {k: v for k, v in r.items() if k != 'modelo' and k != 'pred'}
          for n, r in resultados.items()}
salvar['melhor'] = melhor_nome
with open(f'{PASTA_RESULTADOS}/05_metricas_modelos.json', 'w',
          encoding='utf-8') as f:
    json.dump(salvar, f, indent=2)

## 6. Classifica os clusters DUVIDOSOS com o melhor modelo

Para cada contrato cujo cluster você marcou como `duvidoso`, aplicamos o
melhor modelo. Cada contrato recebe:
- **`pred`**: 1 (eng_obra) ou 0 (nao)
- **`prob_eng_obra`**: probabilidade do classificador

Quanto mais próximo de 0 ou 1, mais confiante. Próximo de 0.5 = limítrofe.

In [ ]:
mask_duv = df['rotulo_humano'] == 'duvidoso'
print(f'Duvidosos a classificar: {mask_duv.sum():,}')

if mask_duv.any():
    X_duv = X_emb[mask_duv.values]
    pred_duv = melhor.predict(X_duv)
    prob_duv = melhor.predict_proba(X_duv)[:, 1]

    df['pred_modelo'] = -1
    df['prob_eng_obra'] = np.nan
    df.loc[mask_duv, 'pred_modelo'] = pred_duv
    df.loc[mask_duv, 'prob_eng_obra'] = prob_duv

    # Propaga rotulos certeiros para colunas finais
    df['classe_final'] = df['rotulo_humano'].map(
        {'eng_obra': 'eng_obra', 'nao': 'nao'})
    df.loc[mask_duv, 'classe_final'] = np.where(
        pred_duv == 1, 'eng_obra_predito', 'nao_predito')

    print('\nDistribuição final (após classificação dos duvidosos):')
    print(df['classe_final'].value_counts().to_string())

    # Top duvidosos classificados como eng_obra (alta confiança)
    duv_eng = (df[mask_duv & (df['pred_modelo'] == 1)]
                .sort_values('prob_eng_obra', ascending=False))
    print(f'\nTop 15 duvidosos classificados como eng_obra '
          f'(maior prob = mais confiante):')
    print(duv_eng[['numeroControlePNCP', 'text', 'cluster',
                    'prob_eng_obra']].head(15).to_string(index=False))
else:
    print('Nenhum cluster duvidoso. Nada a classificar.')
    df['classe_final'] = df['rotulo_humano'].map(
        {'eng_obra': 'eng_obra', 'nao': 'nao'})

## 7. Visualização UMAP 2D — validação visual

Projetamos os embeddings em 2D com UMAP. **Cores:**

- 🟢 **verde** = certeiro `eng_obra` (rotulado por você)
- 🔴 **vermelho** = certeiro `nao`
- 🟢 claro = duvidoso classificado como `eng_obra`
- 🔴 claro = duvidoso classificado como `nao`

**Validação visual:** se os duvidosos classificados como eng (verde claro)
estão no meio dos certeiros eng (verde escuro), o modelo concorda
geometricamente. Outliers indicam casos para revisar.

In [ ]:
# 7.1 UMAP 2D (subsample se for muito grande)
import umap

MAX_VIZ = 8000
if len(df) > MAX_VIZ:
    idx_viz = np.random.RandomState(42).choice(len(df), size=MAX_VIZ,
                                                  replace=False)
else:
    idx_viz = np.arange(len(df))

print(f'Projetando {len(idx_viz):,} pontos em 2D com UMAP...')
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine',
                    random_state=42)
emb_2d = reducer.fit_transform(X_emb[idx_viz])
print('UMAP OK')

In [ ]:
# 7.2 Scatter colorido
df_viz = df.iloc[idx_viz].copy().reset_index(drop=True)
df_viz['x'] = emb_2d[:, 0]
df_viz['y'] = emb_2d[:, 1]

paleta = {
    'eng_obra':          '#1b7837',   # verde escuro — certeiro eng
    'nao':               '#b2182b',   # vermelho escuro — certeiro nao
    'eng_obra_predito':  '#7fbc41',   # verde claro — duvidoso → eng
    'nao_predito':       '#ef8a62',   # vermelho claro — duvidoso → nao
}

fig, ax = plt.subplots(figsize=(11, 8))
for classe, cor in paleta.items():
    sub = df_viz[df_viz['classe_final'] == classe]
    if not len(sub):
        continue
    ax.scatter(sub['x'], sub['y'], c=cor, s=8, alpha=0.55,
               label=f'{classe} (n={len(sub)})', edgecolors='none')
ax.legend(loc='best', frameon=True)
ax.set_title(f'UMAP 2D — classe final dos contratos (modelo: {melhor_nome})')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/07_umap_classe_final.png', dpi=120,
            bbox_inches='tight')
plt.show()

## 8. Veredito final do LLM nos classificados como engenharia

Para os top-N contratos classificados como `eng_obra` (incluindo duvidosos
preditos), o LLM revisa e dá veredito independente. Funciona como **dupla
checagem** antes da análise de rito.

In [ ]:
SYS_VEREDITO = '''Você é analista de contratações públicas brasileiras
(Lei 14.133/2021). Classifique APENAS o TIPO do contrato pelo objeto.

Considere SOMENTE engenharia (CREA/ART) e obras. Arquitetura (CAU/RRT) NÃO
entra: serviço puramente arquitetônico é "nao".

REGRA DE OURO — NÃO são engenharia/obras (são "nao"):
- LOCAÇÃO/ALUGUEL de equipamento
- AQUISIÇÃO/FORNECIMENTO de bens ou materiais
- PARTICIPAÇÃO em evento/curso/workshop
- Serviços administrativos, jurídicos, contábeis, culturais
- Plano/estudo meramente administrativo

São engenharia/obras quando há EXECUÇÃO de serviço técnico ou intervenção
física construtiva.

Responda APENAS no JSON:
{"classe": "eng_obra"|"nao",
 "confianca": 0.0-1.0,
 "motivo": "até 40 palavras"}'''

# Pega os classificados como eng_obra (certeiros + duvidosos preditos eng)
mask_eng = df['classe_final'].isin(['eng_obra', 'eng_obra_predito'])
N_VEREDITO = min(50, mask_eng.sum())
candidatos = df[mask_eng].copy()
# Prioriza os DUVIDOSOS preditos eng (são onde o LLM agrega mais valor)
candidatos['prio'] = (candidatos['classe_final'] == 'eng_obra_predito').astype(int)
candidatos = candidatos.sort_values(['prio', 'prob_eng_obra'],
                                     ascending=[False, False]).head(N_VEREDITO)

print(f'Validando {len(candidatos)} contratos via LLM...')
veredictos = []
for i, (_, row) in enumerate(candidatos.reset_index().iterrows()):
    if i > 0:
        time.sleep(INTERVALO_LLM_SEG)
    obj = str(row['text'])[:600]
    resp = llm_task(MODELO_LLM, SYS_VEREDITO, f'Objeto: {obj}')
    p = extrair_json(resp) or {}
    veredictos.append({
        'numeroControlePNCP': row.get('numeroControlePNCP', ''),
        'cluster': row['cluster'],
        'classe_final': row['classe_final'],
        'prob_modelo': round(float(row.get('prob_eng_obra') or 1.0), 3),
        'llm_classe': p.get('classe', '?'),
        'llm_confianca': float(p.get('confianca', 0) or 0),
        'llm_motivo': str(p.get('motivo', ''))[:300],
        'objeto': obj[:200],
    })
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(candidatos)}')

df_veredictos = pd.DataFrame(veredictos)
df_veredictos['confirmado_eng'] = (
    (df_veredictos['llm_classe'] == 'eng_obra') &
    (df_veredictos['llm_confianca'] >= 0.6))
n_conf = int(df_veredictos['confirmado_eng'].sum())
print(f'\nLLM confirmou {n_conf}/{len(df_veredictos)} como eng_obra.')
df_veredictos.to_csv(f'{PASTA_RESULTADOS}/08_veredito_llm.csv',
                     index=False, encoding='utf-8-sig')
df_veredictos.head(20)

## 9. Análise de rito — baixa PDFs dos confirmados

Os PDFs **não existem ainda** — eles são baixados aqui, apenas para os
contratos **confirmados como engenharia** pelo LLM (Etapa 8). Use
`MAX_PDFS_BAIXAR` para limitar o volume (cada contrato leva ~5–20s).

**Por que buscar na COMPRA e não no contrato?** O `numeroControlePNCP` tem
dígito 1 = contratação (compra) e 2 = contrato. O TR/Projeto Básico ficam
anexados à **compra de origem**, referenciada pelo campo
`numeroControlePNCPCompra` (ou resolvida via API quando o campo não veio
na coleta).

**O que detectamos em cada PDF:**
1. **Marcadores legais (regex):** ART, CREA, Eng. Responsável, Atestado Cap.
   Técnica, Projeto Básico, Obra/Serv. Engenharia, ABNT, Lei 14.133.
2. **Veredito LLM** sobre o trecho do TR/Edital.

**Veredito final do rito:**
- `subenquadramento_real` — PDF legível, SEM marcadores e LLM nega rito →
  **provável violação da Lei 14.133/2021**
- `rotulacao_incorreta_processo_ok` — ≥2 marcadores OU LLM confirma rito
- `indeterminado_*` — compra não resolvida / sem documento / PDF ilegível

In [ ]:
# 9.1 Baixa PDFs dos confirmados (com limite) — via COMPRA de origem
import requests

MAX_PDFS_BAIXAR = 50        # nº máx. de CONTRATOS a processar (limite de tempo)
MAX_DOCS_POR_CONTRATO = 3   # nº máx. de PDFs por contrato

CACHE_PDFS = Path(PASTA_RESULTADOS) / 'cache_pdfs'
CACHE_PDFS.mkdir(parents=True, exist_ok=True)

_API = 'https://pncp.gov.br/api/pncp/v1'
_HEADERS = {'User-Agent': 'Mozilla/5.0 (pesquisa academica PNCP)'}
_RX_NCP = re.compile(r'^(?P<cnpj>\d{14})-(?P<tipo>\d+)-(?P<seq>\d+)/(?P<ano>\d{4})$')

def _decompor_ncp(num):
    m = _RX_NCP.match(str(num).strip()) if num else None
    if not m:
        return None
    return {'cnpj': m['cnpj'], 'tipo': int(m['tipo']),
            'ano': int(m['ano']), 'sequencial': int(m['seq'])}

def _resolver_compra(ncp_contrato, row):
    """Devolve info da COMPRA de origem (cnpj/ano/seq) ou None."""
    # 1) campo salvo na coleta (versões novas de pncp.coleta)
    ncp_compra = row.get('numeroControlePNCPCompra')
    if isinstance(ncp_compra, str) and ncp_compra.strip():
        info = _decompor_ncp(ncp_compra)
        if info:
            return info
    info_c = _decompor_ncp(ncp_contrato)
    if not info_c:
        return None
    # 2) o próprio NCP já é compra (dígito 1)
    if info_c['tipo'] == 1:
        return info_c
    # 3) consulta o detalhe do contrato na API
    url = (f'{_API}/orgaos/{info_c["cnpj"]}/contratos/'
           f'{info_c["ano"]}/{info_c["sequencial"]}')
    try:
        r = requests.get(url, timeout=20, headers=_HEADERS)
        if r.status_code != 200:
            return None
        d = r.json()
        if d.get('numeroControlePNCPCompra'):
            return _decompor_ncp(d['numeroControlePNCPCompra'])
        seqc, anoc = d.get('sequencialCompra'), d.get('anoCompra')
        cnpjc = (d.get('orgaoEntidade') or {}).get('cnpj') or info_c['cnpj']
        if seqc and anoc:
            return {'cnpj': cnpjc, 'tipo': 1, 'ano': int(anoc),
                    'sequencial': int(seqc)}
    except Exception:
        return None
    return None

def _listar_docs_compra(info):
    url = (f'{_API}/orgaos/{info["cnpj"]}/compras/'
           f'{info["ano"]}/{info["sequencial"]}/arquivos')
    try:
        r = requests.get(url, timeout=20, headers=_HEADERS)
        if r.status_code != 200:
            return []
        d = r.json()
        return d if isinstance(d, list) else d.get('data', [])
    except Exception:
        return []

def _baixar_doc(d, info, destino):
    """URL do payload primeiro; fallback: endpoint /arquivos/{seq}."""
    candidatas = [d[c] for c in ('url', 'uri', 'urlArquivo', 'link')
                  if d.get(c)]
    sq = (d.get('sequencialDocumento') or d.get('sequencial')
          or d.get('sequencialArquivo') or d.get('id'))
    if sq:
        candidatas.append(f'{_API}/orgaos/{info["cnpj"]}/compras/'
                          f'{info["ano"]}/{info["sequencial"]}/arquivos/{sq}')
    for u in candidatas:
        try:
            r = requests.get(u, timeout=60, stream=True, headers=_HEADERS)
            if r.status_code == 200:
                with open(destino, 'wb') as f:
                    for ch in r.iter_content(8192):
                        f.write(ch)
                if destino.stat().st_size > 0:
                    return True
        except Exception:
            continue
    return False

# ── Loop de download ─────────────────────────────────────────────────────
confirmados_ncp = (df_veredictos[df_veredictos['confirmado_eng']]
                   ['numeroControlePNCP'].dropna().astype(str).tolist())
confirmados_ncp = confirmados_ncp[:MAX_PDFS_BAIXAR]
print(f'Baixando PDFs de {len(confirmados_ncp)} contratos confirmados '
      f'(limite={MAX_PDFS_BAIXAR})...\n')

df_idx = df.set_index(df['numeroControlePNCP'].astype(str))
pdfs_por_ncp = {}      # ncp_contrato -> [paths de PDF]
status_por_ncp = {}    # ncp_contrato -> 'ok' | 'sem_compra' | 'sem_documento'
n_baixados = n_cache = 0

for i, ncp in enumerate(confirmados_ncp):
    row = df_idx.loc[ncp] if ncp in df_idx.index else {}
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    info_compra = _resolver_compra(ncp, row)
    if not info_compra:
        status_por_ncp[ncp] = 'sem_compra'
        continue
    docs = _listar_docs_compra(info_compra)
    if i < 5:
        print(f'  • {ncp} → compra {info_compra["cnpj"]}-1-'
              f'{info_compra["sequencial"]:06d}/{info_compra["ano"]} '
              f'docs={len(docs)}')
    if not docs:
        status_por_ncp[ncp] = 'sem_documento'
        continue
    arquivos = []
    for d in docs[:MAX_DOCS_POR_CONTRATO]:
        sq = (d.get('sequencialDocumento') or d.get('sequencial')
              or d.get('id') or len(arquivos))
        destino = CACHE_PDFS / f'{ncp.replace("/", "_")}_{sq}.pdf'
        if destino.exists() and destino.stat().st_size > 0:
            n_cache += 1
            arquivos.append(destino)
            continue
        if _baixar_doc(d, info_compra, destino):
            n_baixados += 1
            arquivos.append(destino)
            time.sleep(0.2)
    pdfs_por_ncp[ncp] = arquivos
    status_por_ncp[ncp] = 'ok' if arquivos else 'sem_documento'
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(confirmados_ncp)} | baixados={n_baixados} '
              f'cache={n_cache}')

print(f'\nDownload concluído: {n_baixados} novos, {n_cache} do cache.')
print('Status:', dict(Counter(status_por_ncp.values())))

In [ ]:
# 9.2 Marcadores legais (regex de alta precisão — só CREA/ART, sem CAU/RRT)
MARCADORES = {
    'ART': [r'\banota[çc][ãa]o\s+de\s+responsabilidade\s+t[ée]cnica\b',
            r'\bART\b(?:\s+do\s+CREA)?'],
    'CREA': [r'\bCREA[/\s\-]?\w{0,2}\b',
             r'\bConselho\s+Regional\s+de\s+Engenharia\b'],
    'ENGENHEIRO_RESPONSAVEL': [r'\bengenheir[oa]?\s+respons[áa]vel\b',
                                r'\brespons[áa]vel\s+t[ée]cnico\b'],
    'ATESTADO_CAP_TECNICA': [r'\batestado\s+de\s+capacidade\s+t[ée]cnica\b'],
    'PROJETO_BASICO': [r'\bprojeto\s+b[áa]sico\b',
                        r'\bprojeto\s+executivo\b',
                        r'\bmemorial\s+descritivo\b'],
    'OBRA_SERVICO_ENGENHARIA': [r'\bobra\s+de\s+engenharia\b',
                                  r'\bservi[çc]os?\s+de\s+engenharia\b'],
    'ABNT_NORMA': [r'\bABNT\s+NBR\s*\d+'],
    'LEI_14133_ENGENHARIA': [r'\bart\.?\s*6[°º]?,?\s*(?:inc\.?\s*)?XII\b',
                              r'\bart\.?\s*6[°º]?,?\s*(?:inc\.?\s*)?XX(?:I+)?\b'],
}
_RX_MARC = {n: [re.compile(p, re.IGNORECASE) for p in ps]
            for n, ps in MARCADORES.items()}

def detectar_marcadores(texto):
    if not texto:
        return {n: False for n in MARCADORES}
    return {n: any(rx.search(texto) for rx in lst)
            for n, lst in _RX_MARC.items()}

def extrair_texto_pdf(caminho, max_paginas=30):
    try:
        import fitz
        doc = fitz.open(caminho)
        textos = []
        for i, p in enumerate(doc):
            if i >= max_paginas:
                break
            textos.append(p.get_text())
        doc.close()
        s = '\n'.join(textos)
        # de-hifeniza quebra de linha
        s = re.sub(r'-\s*\n\s*', '', s)
        s = re.sub(r'(?<!\n)\n(?!\n)', ' ', s)
        return re.sub(r'[ \t]+', ' ', s)
    except Exception as e:
        return ''

In [ ]:
# 9.3 Analisa os PDFs baixados: marcadores regex + veredito LLM
SYS_RITO_PDF = '''Você analisa um TRECHO de Termo de Referência / Projeto
Básico / Edital de uma contratação pública. Diga se o documento evidencia
RITO DE ENGENHARIA: exigência de ART do CREA, projeto básico/executivo,
engenheiro responsável, atestado de capacidade técnica, memorial descritivo,
planilha orçamentária, normas ABNT NBR, ou citação dos artigos 6º XII/XX/XXI
da Lei 14.133/2021.

Responda APENAS no JSON:
{"rito_engenharia": "sim"|"nao"|"parcial",
 "evidencias": ["evidência 1", "evidência 2"],
 "confianca": 0.0-1.0}'''

resultados_rito = []
n_llm = 0
for ncp in confirmados_ncp:
    reg = {'numeroControlePNCP': ncp,
           'n_pdfs': len(pdfs_por_ncp.get(ncp, [])),
           'chars_total': 0, 'mk_score': 0, 'llm_rito': '',
           'llm_evidencias': '', 'llm_confianca': 0.0}
    for c in MARCADORES:
        reg[f'mk_{c}'] = False

    st = status_por_ncp.get(ncp, 'sem_compra')
    if st == 'sem_compra':
        reg['classificacao_rito'] = 'indeterminado_sem_compra'
        resultados_rito.append(reg)
        continue
    if st == 'sem_documento' or not pdfs_por_ncp.get(ncp):
        reg['classificacao_rito'] = 'indeterminado_sem_documento'
        resultados_rito.append(reg)
        continue

    textos = [extrair_texto_pdf(a) for a in pdfs_por_ncp[ncp]]
    texto_total = '\n'.join(t for t in textos if t)
    reg['chars_total'] = len(texto_total)
    if reg['chars_total'] < 300:
        reg['classificacao_rito'] = 'indeterminado_pdf_ilegivel'
        resultados_rito.append(reg)
        continue

    # Marcadores regex
    marc = detectar_marcadores(texto_total)
    for c, v in marc.items():
        reg[f'mk_{c}'] = bool(v)
    reg['mk_score'] = sum(1 for v in marc.values() if v)

    # Veredito LLM sobre o trecho
    if n_llm > 0:
        time.sleep(INTERVALO_LLM_SEG)
    resp = llm_task(MODELO_LLM, SYS_RITO_PDF,
                    f'Documento:\n{texto_total[:5000]}')
    n_llm += 1
    pr = extrair_json(resp) or {}
    reg['llm_rito'] = pr.get('rito_engenharia', '')
    reg['llm_evidencias'] = '; '.join(pr.get('evidencias', []))[:300]
    reg['llm_confianca'] = float(pr.get('confianca', 0) or 0)

    # Veredito final: regex OU LLM confirmam o rito
    if reg['mk_score'] >= 2 or pr.get('rito_engenharia') == 'sim':
        reg['classificacao_rito'] = 'rotulacao_incorreta_processo_ok'
    else:
        reg['classificacao_rito'] = 'subenquadramento_real'
    resultados_rito.append(reg)

df_rito = pd.DataFrame(resultados_rito)
print(f'{len(df_rito)} contratos analisados | '
      f'PDFs legíveis={int((df_rito["chars_total"] >= 300).sum())}')
print('\nDistribuição do veredito de rito:')
print(df_rito['classificacao_rito'].value_counts().to_string())
df_rito.to_csv(f'{PASTA_RESULTADOS}/09_analise_rito.csv',
               index=False, encoding='utf-8-sig')

In [ ]:
# 9.4 Gráfico do veredito de rito
if len(df_rito):
    fig, ax = plt.subplots(figsize=(8, 4))
    cont = df_rito['classificacao_rito'].value_counts()
    cores_r = {
        'subenquadramento_real':            '#d62728',
        'rotulacao_incorreta_processo_ok':  '#ff7f0e',
        'indeterminado_sem_compra':         '#9aa0a6',
        'indeterminado_sem_documento':      '#bdbdbd',
        'indeterminado_pdf_ilegivel':       '#cccccc',
    }
    ax.barh(cont.index.astype(str)[::-1], cont.values[::-1],
            color=[cores_r.get(c, '#1f77b4') for c in cont.index[::-1]])
    for i, v in enumerate(cont.values[::-1]):
        ax.text(v, i, f' {v}', va='center')
    ax.set_title('Veredito de rito (PDFs cacheados)')
    ax.set_xlabel('nº contratos')
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/09_rito_distribuicao.png', dpi=120,
                bbox_inches='tight')
    plt.show()

    n_viol = int((df_rito['classificacao_rito'] == 'subenquadramento_real').sum())
    print(f'\n⚠ {n_viol} prováveis violações da Lei 14.133/2021 '
          f'(eng_obra confirmado + PDF SEM rito de engenharia)')

## 10. Exportação consolidada

Salva:
- `modelo_final.joblib` — o melhor classificador treinado (pronto para
  classificar contratos futuros)
- `embeddings_sbert.npy` — embeddings de toda a amostra (caso queira
  reprocessar)
- `relatorio_modelo.md` — síntese executiva
- todos os CSVs intermediários

In [ ]:
# 10.1 Persiste modelo e artefatos
import joblib, datetime as _dt

joblib.dump(melhor, f'{PASTA_RESULTADOS}/modelo_final.joblib')
print(f'Modelo {melhor_nome} salvo em modelo_final.joblib')

# Salva o df completo com classe_final + probabilidade
cols_export = ['numeroControlePNCP', 'text', 'rotulo', 'cluster',
                'rotulo_humano', 'classe_final', 'prob_eng_obra']
df[cols_export].to_csv(f'{PASTA_RESULTADOS}/10_df_classificado.csv',
                       index=False, encoding='utf-8-sig')
print('df_classificado.csv salvo')

In [ ]:
# 10.2 Relatório markdown final
n_total = len(df)
n_cert_eng = int((df['rotulo_humano'] == 'eng_obra').sum())
n_cert_nao = int((df['rotulo_humano'] == 'nao').sum())
n_duv = int((df['rotulo_humano'] == 'duvidoso').sum())
n_duv_eng = int((df['classe_final'] == 'eng_obra_predito').sum())
n_conf_llm = int(df_veredictos['confirmado_eng'].sum()) if len(df_veredictos) else 0
n_rito_viol = (int((df_rito['classificacao_rito'] == 'subenquadramento_real').sum())
                if len(df_rito) else 0)

linhas = [
    '# Modelo Robusto de Subenquadramento — Relatório',
    f'\n_Gerado em {_dt.datetime.now():%Y-%m-%d %H:%M}_',
    '\n## Base',
    f'- Contratos analisados: **{n_total:,}**',
    '\n## Etapa 2 — Clusterização',
    f'- Clusters gerados (KMeans sobre SBERT): **{K}**',
    f'- Silhouette (amostra): {sil:.3f}',
    '\n## Etapa 3 — Revisão humana',
    f'- Certeiros eng_obra: **{n_cert_eng:,}**',
    f'- Certeiros nao: **{n_cert_nao:,}**',
    f'- Duvidosos: **{n_duv:,}**',
    '\n## Etapa 4 — Perfis (LLM)',
    f'- **eng_obra:** {perfil_eng.get("resumo", "-")}',
    f'- **nao:** {perfil_nao.get("resumo", "-")}',
    '\n## Etapa 5 — Modelos comparados',
    '| modelo | F1 macro | F1 eng_obra |',
    '|---|---|---|',
]
for n, r in resultados.items():
    linhas.append(f'| {n} | {r["f1_macro"]:.3f} | {r["f1_eng"]:.3f} |')
linhas += [
    f'\n🏆 Melhor: **{melhor_nome}**',
    '\n## Etapa 6 — Classificação dos duvidosos',
    f'- Duvidosos classificados como eng_obra: **{n_duv_eng:,}**',
    '\n## Etapa 8 — Veredito LLM',
    f'- Confirmados como eng_obra: **{n_conf_llm}** de {len(df_veredictos)}',
    '\n## Etapa 9 — Rito',
    f'- Prováveis violações (sem rito nos PDFs): **{n_rito_viol}**',
    '\n## Artefatos salvos',
    '- `modelo_final.joblib` — classificador pronto para inferência',
    '- `03_revisao_clusters.csv` — rotulação humana',
    '- `04_perfis_llm.json` — descrições dos perfis',
    '- `05_metricas_modelos.json` — F1 dos 3 modelos',
    '- `08_veredito_llm.csv` — veredito por contrato',
    '- `09_analise_rito.csv` — marcadores + LLM por PDF',
    '- `10_df_classificado.csv` — base completa com classe_final',
    '- gráficos PNG: distribuição, matriz de confusão, UMAP, rito',
]
relatorio = '\n'.join(linhas)
with open(f'{PASTA_RESULTADOS}/relatorio_modelo.md', 'w',
          encoding='utf-8') as f:
    f.write(relatorio)
print(relatorio)

---

## Como reusar o modelo em contratos futuros

```python
import joblib
from sentence_transformers import SentenceTransformer

modelo = joblib.load(f'{PASTA_RESULTADOS}/modelo_final.joblib')
sbert = SentenceTransformer('distiluse-base-multilingual-cased-v1')

def classificar_objeto(texto):
    t_limpo = limpar_boilerplate(texto)
    emb = sbert.encode([t_limpo], normalize_embeddings=True)
    pred = modelo.predict(emb)[0]
    prob = modelo.predict_proba(emb)[0, 1]
    return ('eng_obra' if pred == 1 else 'nao'), prob

# exemplo
classe, prob = classificar_objeto(
    'CONTRATAÇÃO DE EMPRESA PARA PAVIMENTAÇÃO ASFÁLTICA DA RUA X')
print(classe, prob)
```